Introduction:

Since the 18th century, atmospheric carbon dioxide (CO2) concentrations have increased from a baseline of 276ppm to 427ppm. The global oceans absorb and dissolve CO2, which leads to increases in acidity from the reaction with water creating carbonic acid. Excess heat trapped in the atmosphere by the greenhouse effect is also absorbed. Warmer oceans have a decreased ability to retain dissolved gases such as oxygen. With the oceans absorbing more than 30% of anthropogenic CO2 and over 90% of excess planetary heat, the risk that ocean acidification and decreased dissolved oxygen levels pose grow each year. Acute ocean acidification and hypoxia events impact coastal ecosystems heavily, with immobile and calcifying organisms being impacted the most.

While satellites can measure surface biogeochemistry, they struggle to resolve ocean floor conditions. In-situ observational data of sea floor dissolved oxygen and saturated aragonite is expensive to collect and sparse. Additionally, it is impossible to forecast regional conditions and hypoxia or acidification events alone. 

This is why it is important to develop and validate models to help inform regional fisheries and ecosystem management efforts. The Modular Ocean Model - Carbon, Ocean Biogeochemistry and Lower Trophics (MOM6-COBALT) is a coupled physics-biogeochemistry model that may be used for such forecasting.  



Validating model outputs requires direct comparisons between past observations and model hindcasts - model simulations of the past using baseline conditions. The following workflow details the comparison of observational data from NOAA Ecomonitoring vessels, demursal sea gliders, Ocean Observation Initiative measurements, EMOLT lobster cage, and Copperarative Commercial Fisheries Research Fleet data.

The first step to creating an operable python workflow for this comparison is importing the correct modules into a kernel and setting up Paths for export: 

In [3]:
#Very first step is to import these modules and libraries into the satprocessing kernel
import numpy as np
import dask as ds
import earthaccess as ea
import xarray as xr
import sys
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import requests
import pandas as pd
import PyCO2SYS as pyco2
import cartopy.crs as ccrs          # ccrs handles projections
import cartopy.feature as cfeature  # cfeature handles land, ocean, coastlines
import cartopy
from shapely.ops import unary_union
import cartopy.feature as cfeature
from datetime import date
from scipy.spatial import KDTree
from datasets import Dataset
import geopandas as gpd
import rioxarray
from shapely.geometry import mapping
from setuptools import setup, find_packages
import statsmodels.api as sm
import seaborn as sns





print("module import success")


sys.path.append(r'C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\RESOURCES\\python\\utilities\\bootstrap\\')
print("path import success")

import pprint
from environment import bootstrap_environment  
pprint.pprint(bootstrap_environment(preferred="edmund_gable"))
print("success")

from utilities.bootstrap.environment import bootstrap_environment
env = bootstrap_environment(preferred="edmund.gable")
env
print("utilities import success")

#Before this next step, install the library 'resource': conda install -c conda-forge resource
from utilities import get_dates, get_source_file_dates, find_text
find_text("stats")

from utilities import get_prod_files 
get_prod_files("DO", period="DOY")
#"DOY" is climatological 'day of year
#'W' is climatological weekly files

from pathlib import Path

root = Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026")
raw_data = Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data")
model_data = Path(r"Z:\\MOM6\Hindcasts\\BTM-O2-REGRID")
processed_data=Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Processed_Data\comparison")


import functions.py as f

module import success
path import success
⚠ Unrecognized preferred label 'edmund_gable'. Valid options: ['NECMAC04363461.local', 'nefscsatdata', 'guihyde', 'Mac.localdomain', 'gdavis', 'egable']
{'dataset_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/DATASETS'),
 'hostname': 'NECL04363598',
 'lookuptable_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/lookuptables'),
 'metadata_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/metadata'),
 'project_root': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'python_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'satlogs_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/logs/satprocessing'),
 'timestamp': datetime.datetime(2026, 7, 21, 9, 10, 26, 533030),
 'utilities_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RES

ModuleNotFoundError: No module named 'functions'

To compare the MOM6-COBALT outputs with raw data, first get an index of the model grids...

In [ ]:
# 1. create the MOM6-COBALT grid df from one of the hindcast data files. 
df_mom6_raw=pd.read_csv(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data\MOM6_COBALT\2004_01_01.csv")
grid_extent = df_mom6_raw[["lat", "lon"]].drop_duplicates().reset_index(drop=True)

# 2. Create a unique integer ID for every single unique grid cell
grid_extent["grid_id"] = grid_extent.index + 1  # 1-indexed IDs

# 3. Save this file so you don't have to keep rebuilding it
export_path=(r"C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\GitHub\\MOM6-COBALT_2026\\data\\Index\\MOM6_Grid_Extent.csv")
grid_extent.to_csv(export_path, date_format='%Y-%m-%d %H:%M:%S', index=False)


print(f"Grid extent file created with {len(grid_extent)} unique grid IDs.")
print(grid_extent.head())

dsg=grid_extent.set_index(['lat','lon']).to_xarray()
dsg

Next, open a raw observational file (ex: downloaded from an ERRDAP search)

In [ ]:
#1. Open the observational data file 


file=(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data\Hydrographic_Monitoring\errdap_search.nc")
df=open_file(file)

#check to make sure units are in mg/L 
check_units(df, 'dissolved_oxygen')
df=df[df['cruise_id']!='HB2101'] #removed this cruise because the oxygen detector malfunctioned

#2. read in the grid extent
mom6_grid_df=pd.read_csv(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Index\MOM6_Grid_Extent.csv")
